# Week 2 lab: problem framing and data suitability

## Goal

Use a bounded data profile to decide whether the supplied evidence can support
a real operational decision. This notebook supports the written problem brief,
ethical risk note, dataset choice, and suitability checklist.

Do not edit the raw CSV or train a model in this lab.

## Setup

The setup cell finds the project root from either the root folder or the
`notebooks/` directory. It then loads the raw teaching data.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display


def locate_project_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "data" / "raw" / "plant_shift_log.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Project root not found. Start Jupyter from the week02 project folder."
    )


PROJECT_ROOT = locate_project_root(Path.cwd().resolve())
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "plant_shift_log.csv"
df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])

print(f"Project root: {PROJECT_ROOT.name}")
print(f"Data source: {DATA_PATH.relative_to(PROJECT_ROOT)}")
print(f"Shape: {df.shape}")

## Steps

### 1. Improve the request

The plant manager asks: **"Use AI to predict machine problems."**

Before running more code, write a decision-ready version in
`docs/problem_brief.md`. It must name the user, decision, action window, and
consequence. Do not name an algorithm.

### 2. Inspect the unit of observation

Review a bounded sample, the column names, and the data types. In your dataset
choice, explain what one row represents.

In [ ]:
display(df.head(6))

schema = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(dtype) for dtype in df.dtypes],
    "missing": [int(df[column].isna().sum()) for column in df.columns],
})
display(schema)

### 3. Measure coverage

Dataset size alone does not establish suitability. Check the time span, number
of lines, and observations by shift. Then ask whether this period represents
the intended operational use.

Note the difference between *elapsed hours* (last timestamp minus first) and
*hourly periods* (how many distinct hours were recorded). A file that starts at
06:00 and ends at 05:00 the next day spans 23 elapsed hours but contains 24
periods. If one line has more rows than periods, something is repeated.

In [ ]:
coverage = pd.Series({
    "start": df["timestamp"].min(),
    "end": df["timestamp"].max(),
    "elapsed_hours": (
        df["timestamp"].max() - df["timestamp"].min()
    ).total_seconds() / 3600,
    "hourly_periods": df["timestamp"].nunique(),
    "number_of_lines": df["line_id"].nunique(),
    "number_of_shifts": df["shift"].nunique(),
})
display(coverage.to_frame("observed"))
display(pd.crosstab(df["line_id"], df["shift"]))

### 4. Collect quality evidence

Identify duplicates, missing values, and two simple plausibility concerns.
Record what must be checked with the data owner or an engineer. Do not silently
repair the values.

In [ ]:
quality_evidence = pd.Series({
    "duplicate_rows": int(df.duplicated().sum()),
    "missing_values": int(df.isna().sum().sum()),
    "negative_energy_rows": int((df["energy_kwh"] < 0).sum()),
    "temperature_above_100_rows": int((df["motor_temp_c"] > 100).sum()),
})
display(quality_evidence.to_frame("count"))

review_rows = df[
    df.duplicated(keep=False)
    | df.isna().any(axis=1)
    | (df["energy_kwh"] < 0)
    | (df["motor_temp_c"] > 100)
].sort_values(["timestamp", "line_id"])
display(review_rows)

### 5. Check decision fit

Complete the following questions in `docs/data_choice.md` and the suitability
checklist.

1. Does the file contain a verified machine failure or inspection outcome?
2. Can one day represent ordinary variation, different products, and seasonality?
3. Are engineering limits for temperature, downtime, and energy provided?
4. Can the proposed user act on a result before shift handover?
5. Which claim would be supportable now, and which claim would overstate the evidence?

In [ ]:
decision_fit = pd.DataFrame([
    {"need": "Hourly line observations", "present": "Yes", "evidence": "One record per line-hour"},
    {"need": "Verified failure outcome", "present": "No", "evidence": "No failure or inspection field"},
    {"need": "Approved engineering limits", "present": "Unknown", "evidence": "Not supplied with the file"},
    {"need": "Representative history", "present": "No", "evidence": "One day only"},
    {"need": "Permitted teaching use", "present": "Yes", "evidence": "README permits coursework use"},
])
display(decision_fit)

### 6. Map stakeholders

For each role, record what matters, what evidence they need, what failure they
fear, and what authority they hold.

In [ ]:
stakeholder_template = pd.DataFrame([
    {"role": "User", "candidate": "Maintenance supervisor", "value": "", "risk": "", "authority": ""},
    {"role": "Decision owner", "candidate": "Plant or maintenance manager", "value": "", "risk": "", "authority": ""},
    {"role": "Affected people", "candidate": "Operators and technicians", "value": "", "risk": "", "authority": ""},
    {"role": "Data owner", "candidate": "Plant organisation", "value": "", "risk": "", "authority": ""},
])
display(stakeholder_template)

### 7. Reach a preliminary decision

Choose **Proceed**, **Pilot**, **Change question**, or **Stop**. Support the
choice with evidence from the notebook. A strong answer may narrow the original
request rather than force the data to support it.

## Checks

Run the final cell. These checks confirm that the teaching file loaded as
expected. They do not prove that the data are suitable for deployment.

In [ ]:
assert df.shape == (73, 8)
assert df["line_id"].nunique() == 3
assert df["timestamp"].nunique() == 24
assert df.duplicated().sum() == 1
assert df["downtime_min"].isna().sum() == 1
assert (df["energy_kwh"] < 0).sum() == 1
assert (df["motor_temp_c"] > 100).sum() == 1
print("Notebook checks passed. Complete the four written outputs before committing.")

## Next steps

Complete the four files in `docs/`, save this executed notebook, inspect
`git status`, and create the Week 2 commit. Bring the dataset source, licence,
and your provisional suitability decision to Week 3.